# Intermediate ML Demo: Customer Segmentation with K-Means

**Platform:** Amazon SageMaker AI Notebooks (JupyterLab)  
**Algorithm:** K-Means clustering (unsupervised learning)  
**Business scenario:** An online retailer wants to group customers by purchasing behavior to design more relevant engagement campaigns.

## Learning objectives
By completing this notebook, you will:
1. Generate and inspect a realistic, self-contained customer dataset.
2. Select numeric features and standardize them for distance-based clustering.
3. Compare cluster counts using the elbow method and silhouette score.
4. Train K-Means, visualize clusters with PCA, and profile segments.
5. Turn cluster profiles into business actions and assign a segment to a new customer.
6. Save the scaler and clustering model for later reuse.

> **Data note:** The notebook generates synthetic customer data so the lab works without downloading a public dataset or exposing real customer information. The results are educational, not production findings. The model runs directly in the notebook; no SageMaker training job or hosted endpoint is created.

## 1. Import libraries and set reproducibility

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("Libraries are ready.")

## 2. Create a synthetic customer dataset

Each row represents one customer. Features combine demographic context and shopping behavior. In a real project, obtain approved data from your source system and document the meaning, quality, and permitted use of every feature.

The dataset is generated from several plausible behavior patterns, but the actual segment label is **not** included as a model feature. Clustering is unsupervised: the algorithm finds groups using similarities in the input data.

In [ ]:
def clipped_normal(rng, mean, sd, n, low=None, high=None):
    values = rng.normal(mean, sd, n)
    if low is not None or high is not None:
        values = np.clip(values, low if low is not None else -np.inf,
                         high if high is not None else np.inf)
    return values

rng = np.random.default_rng(RANDOM_STATE)

# Approximate behavior patterns used only to create a teaching dataset.
# These profile names are not added to the generated customer records.
profiles = [
    # income, spend score, age, orders/90d, avg order value, recency, visits/month, discount rate, size
    (88, 84, 39, 22, 3600, 8, 18, 0.14, 240),   # active high value
    (38, 57, 32, 18, 950, 12, 14, 0.64, 270),   # promotion sensitive
    (98, 27, 49, 3, 5100, 78, 4, 0.12, 220),    # affluent low engagement
    (56, 86, 24, 16, 1650, 10, 24, 0.26, 250),  # digital trend shopper
    (74, 49, 43, 7, 2950, 91, 6, 0.49, 220),   # lapsed former value
]

rows = []
customer_num = 1
for income, spend, age, orders, aov, recency, visits, discount, size in profiles:
    n = size
    rows.append(pd.DataFrame({
        "Customer_ID": [f"C{customer_num + i:04d}" for i in range(n)],
        "Annual_Income_k": clipped_normal(rng, income, 11, n, 18, 150),
        "Spending_Score": clipped_normal(rng, spend, 9, n, 1, 100),
        "Age": clipped_normal(rng, age, 7, n, 18, 80),
        "Orders_Last_90_Days": np.rint(clipped_normal(rng, orders, 3.5, n, 0, 30)).astype(int),
        "Average_Order_Value": clipped_normal(rng, aov, max(150, aov * 0.18), n, 150, 10000),
        "Days_Since_Last_Purchase": np.rint(clipped_normal(rng, recency, 12 if recency > 50 else 5, n, 0, 180)).astype(int),
        "Website_Visits_Per_Month": np.rint(clipped_normal(rng, visits, 3.5, n, 0, 60)).astype(int),
        "Discount_Usage_Rate": clipped_normal(rng, discount, 0.09, n, 0, 1),
    }))
    customer_num += n

customers = pd.concat(rows, ignore_index=True)
customers = customers.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
customers["Annual_Income_k"] = customers["Annual_Income_k"].round(1)
customers["Spending_Score"] = customers["Spending_Score"].round(1)
customers["Age"] = customers["Age"].round().astype(int)
customers["Average_Order_Value"] = customers["Average_Order_Value"].round(2)
customers["Discount_Usage_Rate"] = customers["Discount_Usage_Rate"].round(2)

# Save a copy in the notebook's current working directory for optional reuse.
csv_path = "customer_segmentation_sample.csv"
customers.to_csv(csv_path, index=False)

print(f"Rows: {len(customers):,}; columns: {len(customers.columns)}")
display(customers.head(10))
print("Saved synthetic data to:", csv_path)

## 3. Inspect data quality and explore customer behavior

In [ ]:
print("Column types and non-null counts:")
customers.info()

print("\nMissing values per column:")
display(customers.isna().sum().to_frame("Missing_Count"))

print("\nDescriptive statistics:")
display(customers.describe().T.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=customers, x="Annual_Income_k", y="Spending_Score", ax=axes[0], alpha=0.65)
axes[0].set_title("Annual income vs. spending score")
axes[0].set_xlabel("Annual income (thousands of currency units)")
sns.scatterplot(data=customers, x="Days_Since_Last_Purchase", y="Orders_Last_90_Days", ax=axes[1], alpha=0.65)
axes[1].set_title("Recency vs. recent order frequency")
axes[1].set_xlabel("Days since last purchase")
axes[1].set_ylabel("Orders in the last 90 days")
plt.tight_layout()
plt.show()

## 4. Choose features and standardize them

K-Means uses distances between rows. Features with larger numeric ranges (for example, average order value) can dominate features with smaller ranges (for example, a discount rate) if they are not scaled. `StandardScaler` transforms each feature to approximately mean 0 and standard deviation 1.

We exclude `Customer_ID` because it is an identifier, not customer behavior. The target label is also absent because this is an unsupervised learning problem.

In [ ]:
feature_columns = [
    "Annual_Income_k",
    "Spending_Score",
    "Age",
    "Orders_Last_90_Days",
    "Average_Order_Value",
    "Days_Since_Last_Purchase",
    "Website_Visits_Per_Month",
    "Discount_Usage_Rate",
]

X = customers[feature_columns].copy()

if X.isna().any().any():
    raise ValueError("Missing feature values found. Add an imputation step before scaling.")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features used:", feature_columns)
print("Raw shape:", X.shape)
print("Scaled shape:", X_scaled.shape)
print("First 3 rows after scaling:")
display(pd.DataFrame(X_scaled, columns=feature_columns).head(3).round(2))

## 5. Select the number of clusters: elbow + silhouette

- **Elbow method:** compares K-Means inertia (within-cluster sum of squared distances). Look for a bend where adding more clusters gives diminishing improvement.
- **Silhouette score:** measures how well each point fits within its assigned cluster compared with nearby clusters. Higher is generally better; the score ranges from -1 to 1.

Neither metric chooses the business answer automatically. Use them alongside segment interpretability, stability, and business needs.

In [ ]:
candidate_ks = range(2, 9)
inertias = []
silhouette_scores = []

for k in candidate_ks:
    candidate_model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = candidate_model.fit_predict(X_scaled)
    inertias.append(candidate_model.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

metrics_by_k = pd.DataFrame({
    "K": list(candidate_ks),
    "Inertia": inertias,
    "Silhouette_Score": silhouette_scores,
})
display(metrics_by_k.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(metrics_by_k["K"], metrics_by_k["Inertia"], marker="o")
axes[0].set_title("Elbow method")
axes[0].set_xlabel("Number of clusters (K)")
axes[0].set_ylabel("Inertia (lower is better)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(metrics_by_k["K"], metrics_by_k["Silhouette_Score"], marker="o")
axes[1].set_title("Silhouette analysis")
axes[1].set_xlabel("Number of clusters (K)")
axes[1].set_ylabel("Silhouette score (higher is better)")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

silhouette_best_k = int(metrics_by_k.loc[metrics_by_k["Silhouette_Score"].idxmax(), "K"])
selected_k = 5  # Teaching choice: the elbow and interpretable five retail profiles support K=5 here.
print(f"Highest silhouette score among tested values: K={silhouette_best_k}")
print(f"Selected K for this demo: {selected_k}")
print("Why not automatically choose the silhouette winner? K=2 is only slightly better on this metric, while K=5 preserves actionable customer subgroups and is near the elbow. Review both metrics and business usefulness.")

## 6. Train the final K-Means model

For this synthetic retail scenario, choose K=5 after comparing both metrics: K=2 has the highest silhouette score, but K=5 has a score close to it and preserves five useful customer behavior profiles. The value is selected explicitly in the cell below, rather than silently letting a single metric decide. For real data, compare nearby values, cluster stability, segment interpretability, and stakeholder needs before finalizing K.

In [ ]:
kmeans = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=10)
customers["Cluster"] = kmeans.fit_predict(X_scaled)

print("Final K:", selected_k)
print("Cluster counts:")
display(customers["Cluster"].value_counts().sort_index().rename("Customer_Count").to_frame())

## 7. Visualize clusters using PCA

In [ ]:
# PCA compresses the standardized feature space into two dimensions for visualization.
# Clustering itself was performed on all selected scaled features, not on this 2D projection.
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
plot_data = pd.DataFrame({
    "PCA_1": X_pca[:, 0],
    "PCA_2": X_pca[:, 1],
    "Cluster": customers["Cluster"].astype(str).values,
})

plt.figure(figsize=(10, 7))
sns.scatterplot(data=plot_data, x="PCA_1", y="PCA_2", hue="Cluster", palette="tab10", alpha=0.75, s=45)
plt.title("Customer clusters projected into two dimensions with PCA")
plt.xlabel(f"Principal component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"Principal component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

print(f"Variance retained by the two plotted PCA components: {pca.explained_variance_ratio_.sum():.1%}")
print("PCA is only a visualization here; the K-Means model used the full scaled feature set.")

## 8. Profile each cluster in business terms

Cluster IDs (0, 1, 2, ...) are arbitrary. Do not assume Cluster 0 is better than Cluster 1. Inspect the customer count and typical feature values to understand what differentiates each group.

The median is used for the profile table because it is less sensitive to unusually high order values or purchase counts.

In [ ]:
cluster_counts = customers.groupby("Cluster").size().rename("Customer_Count")
cluster_profile = customers.groupby("Cluster")[feature_columns].median().round(2)
cluster_profile.insert(0, "Customer_Count", cluster_counts)
cluster_profile["Customer_Percentage"] = (cluster_profile["Customer_Count"] / len(customers) * 100).round(1)

display(cluster_profile.sort_values("Customer_Count", ascending=False))

# Compare standardized means to identify which features distinguish the clusters.
scaled_for_profile = pd.DataFrame(X_scaled, columns=feature_columns, index=customers.index)
scaled_for_profile["Cluster"] = customers["Cluster"]
standardized_profile = scaled_for_profile.groupby("Cluster")[feature_columns].mean().round(2)
plt.figure(figsize=(13, max(4, selected_k * 0.8)))
sns.heatmap(standardized_profile, annot=True, center=0, cmap="vlag", fmt=".2f")
plt.title("Cluster signatures: mean standardized feature values")
plt.tight_layout()
plt.show()

## 9. Give the clusters provisional business labels

This section adds transparent, rule-based labels based on the median profile. These rules are **interpretation aids**, not part of K-Means training. Review the profiles and adjust thresholds for the retailer's business context. If a rule does not fit a real dataset, edit it rather than forcing the labels.

In [ ]:
def label_segment(profile):
    income = profile["Annual_Income_k"]
    spend = profile["Spending_Score"]
    age = profile["Age"]
    orders = profile["Orders_Last_90_Days"]
    aov = profile["Average_Order_Value"]
    recency = profile["Days_Since_Last_Purchase"]
    visits = profile["Website_Visits_Per_Month"]
    discount = profile["Discount_Usage_Rate"]

    if income >= 80 and spend < 40:
        return "Affluent but Under-Engaged"
    if recency >= 55 and orders <= 10:
        return "At-Risk / Lapsed Customers"
    if spend >= 70 and orders >= 12 and age <= 32 and visits >= 18:
        return "Digital Trend Shoppers"
    if spend >= 70 and orders >= 12:
        return "High-Value Active Shoppers"
    if discount >= 0.50 and orders >= 10:
        return "Promotion-Sensitive Regulars"
    if aov >= 2500 and spend >= 40:
        return "Steady Higher-Value Shoppers"
    return "Steady / Mixed-Value Shoppers"

segment_names = cluster_profile.apply(label_segment, axis=1).to_dict()
cluster_profile_named = cluster_profile.copy()
cluster_profile_named["Provisional_Segment_Name"] = cluster_profile_named.index.map(segment_names)

display(cluster_profile_named.sort_values("Customer_Count", ascending=False))
customers["Segment_Name"] = customers["Cluster"].map(segment_names)
print("Reminder: segment names are explanatory labels assigned after clustering; they are not learned classes.")
display(customers[["Customer_ID", "Cluster", "Segment_Name"]].head(10))

## 10. Translate segments into marketing actions

Treat these as starting hypotheses. Validate them through controlled campaigns and measure conversion, incremental revenue, retention, margin, and opt-outs. Avoid blanket discounting where customers already purchase frequently.

In [ ]:
recommendations = {
    "High-Value Active Shoppers": "Reward loyalty, provide early access, and use relevant cross-sell offers rather than unnecessary blanket discounts.",
    "Digital Trend Shoppers": "Test new-product previews, mobile/app engagement, referral offers, and social campaign creatives.",
    "Promotion-Sensitive Regulars": "Use targeted coupons and bundles; measure incremental margin to ensure promotions are profitable.",
    "Affluent but Under-Engaged": "Test premium recommendations and personalized journeys; investigate why purchase frequency is low.",
    "At-Risk / Lapsed Customers": "Run a measured win-back journey, remind customers of relevant products, and review churn drivers.",
    "Steady Higher-Value Shoppers": "Use complementary bundles and tailored recommendations to increase repeat purchases.",
    "Steady / Mixed-Value Shoppers": "Test relevant product recommendations and onboarding content to build purchase frequency.",
}

segment_summary = cluster_profile_named.reset_index()[["Cluster", "Customer_Count", "Customer_Percentage", "Provisional_Segment_Name"]].copy()
segment_summary["Suggested_Action"] = segment_summary["Provisional_Segment_Name"].map(recommendations)
display(segment_summary.sort_values("Customer_Count", ascending=False))

largest_segment = segment_summary.loc[segment_summary["Customer_Count"].idxmax()]
print(f"Largest segment: {largest_segment['Provisional_Segment_Name']} "
      f"({int(largest_segment['Customer_Count'])} customers; {largest_segment['Customer_Percentage']:.1f}%).")

## 11. Assign a segment to a new customer

A new customer's features must use the same column names, order, units, and preprocessing used during training. The saved scaler transforms the new row; the trained K-Means model predicts its cluster; the lookup adds the provisional business label.

In [ ]:
new_customer = pd.DataFrame([{
    "Annual_Income_k": 92.0,
    "Spending_Score": 78.0,
    "Age": 36,
    "Orders_Last_90_Days": 20,
    "Average_Order_Value": 3400.0,
    "Days_Since_Last_Purchase": 6,
    "Website_Visits_Per_Month": 19,
    "Discount_Usage_Rate": 0.18,
}], columns=feature_columns)

new_customer_scaled = scaler.transform(new_customer)
predicted_cluster = int(kmeans.predict(new_customer_scaled)[0])
predicted_segment = segment_names[predicted_cluster]

print("New customer input:")
display(new_customer)
print("Predicted cluster:", predicted_cluster)
print("Provisional segment:", predicted_segment)
print("Suggested action:", recommendations.get(predicted_segment, "Review the segment profile and design a suitable experiment."))

## 12. Save and reload the reusable model artifact

In [ ]:
model_artifact = {
    "feature_columns": feature_columns,
    "scaler": scaler,
    "kmeans": kmeans,
    "segment_names": segment_names,
    "selected_k": selected_k,
}
artifact_path = "customer_segmentation_model.joblib"
joblib.dump(model_artifact, artifact_path)
print("Saved model artifact:", artifact_path)

# Reload and verify that prediction is reproducible.
loaded_artifact = joblib.load(artifact_path)
loaded_scaled = loaded_artifact["scaler"].transform(new_customer[loaded_artifact["feature_columns"]])
loaded_cluster = int(loaded_artifact["kmeans"].predict(loaded_scaled)[0])
loaded_segment = loaded_artifact["segment_names"][loaded_cluster]
print("Segment predicted by reloaded artifact:", loaded_segment)
assert loaded_cluster == predicted_cluster
assert loaded_segment == predicted_segment
print("Validation passed: saved artifact reproduces the prediction.")

## 13. Discussion: limitations and production extensions

- **Synthetic data:** Patterns are constructed for teaching; real customer groups may overlap or change over time.
- **K-Means assumptions:** It favors compact, roughly spherical clusters and can be affected by outliers and feature selection.
- **Choosing K:** Combine silhouette and elbow analysis with stability checks, business usefulness, and stakeholder review.
- **Scaling:** Use the same fitted scaler for every future customer; never fit a new scaler on inference data.
- **Segment drift:** Re-check segment distributions and campaign outcomes as customer behavior changes.
- **Privacy and responsible use:** Use approved data, minimize personally identifiable information, and review whether each feature is necessary and appropriate.
- **Production architecture:** A next step could read approved customer data from Amazon S3, prepare features, train and track a model, and run batch segmentation on a schedule. This notebook does not create AWS resources beyond the notebook compute already running.

## 14. Student exercises

1. Change the candidate cluster range from `2..8` to `2..12`. Does the silhouette winner change? Is the result more useful to a marketing team?
2. Remove `Age` from `feature_columns`, rerun from the feature preparation cell, and compare the customer profiles.
3. Compare cluster sizes for different `random_state` values. Does the segmentation appear stable?
4. Add a new feature such as `Returns_Rate` and explain how it could change the segments.
5. Pick one segment and propose an A/B test with a success metric and a guardrail metric.

**Instructor checkpoint:** Students should be able to explain why scaling matters, why customer IDs are excluded, why PCA is only a visualization here, and why cluster IDs do not have an inherent ranking.